In [1]:
import sys
import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__)

3.12.6 2.0.2 2.3.3 1.6.1


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 3265 

data = pd.read_csv("data/heart_disease.csv")
feature_names = [c for c in data.columns if c != "disease"]
X, y = data[feature_names].to_numpy(), data["disease"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED
)
print(data.shape, round(y.mean(), 3), X_train.shape, X_test.shape)

(297, 14) 0.461 (207, 13) (90, 13)


SEED (last 4 digits of student ID) = 3265

Q1: Parent Entropy

6 of 12 patients have disease=1, 6 have disease=0. p1=0.5, p0=0.5.

H = −0.5·log2(0.5) − 0.5·log2(0.5) = −0.5·(−1) − 0.5·(−1) = 0.5 + 0.5 = 1.0 bit

Q2:  Root feature selection

age_55_plus:
p1 = 5/6, p2 = 1/6 (group age=1: 6 patients, 5 disease)
p1 = 1/6, p2 = 5/6 (group age=0: 6 patients, 1 disease)

H = -(5/6)*log2(5/6) - (1/6)*log2(1/6)
H = 0.219 + 0.431
h = 0.650

H_after = (6/12)*0.650 + (6/12)*0.650 = 0.650
Gain(age_55_plus) = 1 - 0.650 = 0.350


exercise_angina:
p1 = 4/4, p2 = 0/4 (group angina=1: 4 patients, 4 disease)
p1 = 2/8, p2 = 6/8 (group angina=0: 8 patients, 2 disease)

H(angina=1) = -(4/4)*log2(4/4) - (0/4)*log2(0/4)
h1 = 0

H(angina=0) = -(2/8)*log2(2/8) - (6/8)*log2(6/8)
H = 0.5 + 0.311
h2 = 0.811

H_after = (4/12)*0 + (8/12)*0.811 = 0.541
Gain(exercise_angina) = 1 - 0.541 = 0.459


male:
p1 = 3/6, p2 = 3/6 (group male=1: 6 patients, 3 disease)
p1 = 3/6, p2 = 3/6 (group male=0: 6 patients, 3 disease)

H = -(3/6)*log2(3/6) - (3/6)*log2(3/6)
H = 0.5 + 0.5
h = 1.0

H_after = (6/12)*1.0 + (6/12)*1.0 = 1.0
Gain(male) = 1 - 1.0 = 0


high_bp:
p1 = 4/6, p2 = 2/6 (group bp=1: 6 patients, 4 disease)
p1 = 2/6, p2 = 4/6 (group bp=0: 6 patients, 2 disease)

H = -(4/6)*log2(4/6) - (2/6)*log2(2/6)
H = 0.390 + 0.528
h = 0.918

H_after = (6/12)*0.918 + (6/12)*0.918 = 0.918
Gain(high_bp) = 1 - 0.918 = 0.082


Summary:
age_55_plus      0.350
exercise_angina  0.459
male             0
high_bp          0.082

Root = exercise_angina, highest gain = 0.459

Q3: One level down (inside angina=0 branch, 8 patients, parent H = 0.811)

age_55_plus:
p1 = 1/2, p2 = 1/2 (group age=1: 2 patients, 1 disease)
p1 = 1/6, p2 = 5/6 (group age=0: 6 patients, 1 disease)

H(age=1) = -(1/2)*log2(1/2) - (1/2)*log2(1/2)
h1 = 1.0

H(age=0) = -(1/6)*log2(1/6) - (5/6)*log2(5/6)
H = 0.431 + 0.219
h2 = 0.650

H_after = (2/8)*1.0 + (6/8)*0.650 = 0.738
Gain(age_55_plus | angina=0) = 0.811 - 0.738 = 0.074


male:
p1 = 0/3, p2 = 3/3 (group male=1: 3 patients, 0 disease)
p1 = 2/5, p2 = 3/5 (group male=0: 5 patients, 2 disease)

h1 = 0

H(male=0) = -(2/5)*log2(2/5) - (3/5)*log2(3/5)
H = 0.529 + 0.442
h2 = 0.971

H_after = (3/8)*0 + (5/8)*0.971 = 0.607
Gain(male | angina=0) = 0.811 - 0.607 = 0.204


high_bp:
p1 = 1/3, p2 = 2/3 (group bp=1: 3 patients, 1 disease)
p1 = 1/5, p2 = 4/5 (group bp=0: 5 patients, 1 disease)

H(bp=1) = -(1/3)*log2(1/3) - (2/3)*log2(2/3)
H = 0.528 + 0.390
h1 = 0.918

H(bp=0) = -(1/5)*log2(1/5) - (4/5)*log2(4/5)
H = 0.464 + 0.258
h2 = 0.722

H_after = (3/8)*0.918 + (5/8)*0.722 = 0.795
Gain(high_bp | angina=0) = 0.811 - 0.795 = 0.016


Summary (inside angina=0 branch):
age_55_plus   0.074
male          0.204
high_bp       0.016

Chosen feature = male, gain = 0.204 (vs gain = 0 at the root)

At the root, male split the full 12 patients 3/3 disease in both the male and female 
groups, so by itself it carried no information about disease overall. But once we 
already know angina=0, the remaining 8 patients split differently: all 3 males in this 
subgroup have no disease, while the 2 disease cases are both non-males. The useful 
pattern in male was cancelled out at the root by the opposite pattern among the angina=1 
patients, and only becomes visible once that interaction is removed by conditioning on 
angina first.